In [13]:
# CELL 1 — MASTER SETUP (run this first in every new file/session)
# Copy this entire cell → run → then run any experiment cell

import os, gc, torch, logging, numpy as np, math, re
import pandas as pd, warnings
warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)
import transformers; transformers.logging.set_verbosity_error()

from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, roc_auc_score, matthews_corrcoef)
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from collections import Counter
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup)
import transformers.modeling_utils as mu
try: import emoji
except: os.system('pip install emoji -q'); import emoji

# ── Paths ──────────────────────────────────────────────────
BASE      = './BanglishDetect/'
base      = BASE + 'data/raw/'
proc_base = BASE + 'data/processed/'
res_base  = BASE + 'results/'
seed_base = BASE + 'results/seed_runs/'
for p in [res_base, seed_base, proc_base]:
    os.makedirs(p, exist_ok=True)

# ── GPU ────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU!"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
DEVICE = torch.device('cuda:0')
ver      = tuple(int(x) for x in transformers.__version__.split('.')[:2])
EVAL_KEY = 'eval_strategy' if ver>=(4,46) else 'evaluation_strategy'
SEEDS    = [42, 123, 456]
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Seeds ──────────────────────────────────────────────────
def set_seed(seed):
    import random; random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    os.environ['PYTHONHASHSEED']=str(seed)

# ── Metrics ────────────────────────────────────────────────
def compute_all_metrics(y_true, y_pred, y_prob=None):
    return {
        'accuracy': round(accuracy_score(y_true,y_pred),4),
        'precision':round(precision_score(y_true,y_pred,average='weighted',zero_division=0),4),
        'recall':   round(recall_score(y_true,y_pred,average='weighted',zero_division=0),4),
        'f1':       round(f1_score(y_true,y_pred,average='weighted',zero_division=0),4),
        'mcc':      round(matthews_corrcoef(y_true,y_pred),4),
        'auc':      round(roc_auc_score(y_true,y_prob if y_prob is not None else y_pred),4),
    }
def compute_PIS(f1_pk,f1_p0):
    return round((f1_pk-f1_p0)/f1_p0*100,2) if f1_p0 else 0

# ── Data loaders ───────────────────────────────────────────
def load_split(corpus,condition,split):
    df=pd.read_csv(f"{proc_base}{corpus}_{condition}_{split}.csv").dropna(subset=['text'])
    df['text']=df['text'].astype(str); return df['text'].tolist(),df['label'].tolist()
def load_native_split(condition,split):
    df=pd.read_csv(f"{proc_base}native_{condition}_{split}.csv").dropna(subset=['text'])
    df['text']=df['text'].astype(str); return df['text'].tolist(),df['label'].tolist()
def load_yt_split(condition,split):
    df=pd.read_csv(f"{proc_base}yt_{condition}_{split}.csv").dropna(subset=['text'])
    df['text']=df['text'].astype(str); return df['text'].tolist(),df['label'].tolist()

# ── Emotion resources ──────────────────────────────────────
ALL_EMOTION = set([
    'মরতে','মরে','আত্মহত্যা','মৃত্যু','শেষ','বাঁচতে','কষ্ট','যন্ত্রণা',
    'ব্যথা','কাঁদি','কাঁদছি','কাঁদছে','একা','নিঃসঙ্গ','অসহায়','হতাশ',
    'নিরাশ','বিষণ্ণ','ক্লান্ত','বিধ্বস্ত','ভেঙে','ভেঙেছি','দুঃখ','দুঃখী',
    'খারাপ','অস্থির','চিন্তিত','ভয়','রাগ','সুখী','খুশি','আনন্দ','ভালো',
    'ভালোবাসা','আশা','স্বপ্ন','সফল','আলো','হাসি',
    'kosto','kanna','ekla','mara','shesh','nirash','kharap',
    'mon kharap','valo nei','tension','worry','sad','depressed','lonely',
    'happy','joy','love','hope','smile','bhalo',
])
NEGATION = {'না','নেই','নয়','নি','নিই'}
BENGALI_SW = {
    'এবং','ও','বা','কিন্তু','যদি','তবে','কারণ','যে','যা','এই','ওই',
    'সেই','এটা','ওটা','সেটা','আমি','আমার','আমাকে','তুমি','তোমার',
    'সে','তার','তাকে','এখানে','ওখানে','হয়','হবে','করা','করে',
    'থেকে','দিয়ে','জন্য','মধ্যে','আছে','ছিল','কি','কী','কোন',
    'খুব','অনেক','বেশি',
}
SAFE_SW = BENGALI_SW - ALL_EMOTION - NEGATION
EMOTICONS = {
    '😭':'[CRY]','😢':'[SAD_FACE]','😔':'[PENSIVE]','💔':'[BROKEN_HEART]',
    '😩':'[WEARY]','🥺':'[PLEADING]','☠️':'[SKULL]','💀':'[DEATH]',
    '😞':'[DISAPPOINTED]','😟':'[WORRIED]','😓':'[SWEAT_SAD]',
    '🖤':'[BLACK_HEART]','😊':'[SMILE]','😄':'[HAPPY]',
    '❤️':'[HEART]','😂':'[LAUGH]','🙏':'[PRAYER]',
}

# ── Preprocessing ──────────────────────────────────────────
def _rn(text):
    t=re.sub(r'http\S+|www\S+','',str(text))
    t=re.sub(r'@\w+','',t); t=re.sub(r'#\w+','',t)
    t=re.sub(r'<.*?>','',t); return re.sub(r'\s+',' ',t).strip()

def preprocess_P0(t): return str(t).strip()
def preprocess_P1(t): return _rn(t)
def preprocess_P2(t):
    t=_rn(t); t=re.sub(r'[a-zA-Z]','',t)
    t=re.sub(r'[^\u0980-\u09FF\s]','',t)
    return ' '.join(w for w in t.split() if w not in BENGALI_SW)
def preprocess_P3(t):
    t=preprocess_P2(t); t=re.sub(r'\d+','',t)
    return ' '.join(w for w in t.split() if len(w)>=2)
def _base(text,em=True,neg=True,bn=True,el=True):
    t=str(text)
    if em:
        for e,tag in EMOTICONS.items(): t=t.replace(e,f' {tag} ')
        t=emoji.replace_emoji(t,replace='')
    else:
        for e in EMOTICONS: t=t.replace(e,' ')
        t=emoji.replace_emoji(t,replace='')
    t=_rn(t)
    if el: t=re.sub(r'(.)\1{2,}',r'\1\1',t)
    result=[]
    for tok in t.split():
        if em and tok.startswith('[') and tok.endswith(']'):
            result.append(tok); continue
        if re.match(r'^[^\u0980-\u09FFa-zA-Z\[\]]+$',tok): continue
        if tok in NEGATION:
            if neg: result.append(tok)
            elif tok not in SAFE_SW: result.append(tok)
            continue
        if tok in ALL_EMOTION: result.append(tok); continue
        if re.search(r'[a-zA-Z]',tok):
            if bn and len(tok)>=3: result.append(tok.lower())
            continue
        if tok in SAFE_SW: continue
        if len(tok)>=2: result.append(tok)
    return ' '.join(result)

def preprocess_P4_EPH(t):   return _base(t,True,True,True,True)
EPH_full         = preprocess_P4_EPH
def EPH_no_emoticon(t):  return _base(t,False,True,True,True)
def EPH_no_negation(t):  return _base(t,True,False,True,True)
def EPH_no_banglish(t):  return _base(t,True,True,False,True)
def EPH_no_elongation(t):return _base(t,True,True,True,False)

# ── CESS metrics ───────────────────────────────────────────
def compute_IPR(orig,proc,s=2000):
    def H(texts):
        w=[]
        for t in texts[:s]: w.extend(str(t).split())
        if not w: return 0
        c=Counter(w); tot=sum(c.values())
        return -sum((v/tot)*math.log2(v/tot) for v in c.values())
    h0=H(orig); return round(H(proc)/h0,4) if h0 else 0
def compute_BSR(orig,proc):
    def cb(texts):
        return sum(1 for t in texts for tok in str(t).split()
                   if re.search(r'[a-zA-Z]',tok) and len(tok)>=3)
    o=cb(orig); return round(cb(proc)/o,4) if o else 1.0
def compute_ELPS(orig,proc):
    def ce(texts):
        return sum(1 for t in texts for tok in str(t).split()
                   if tok in ALL_EMOTION or
                   (tok.startswith('[') and tok.endswith(']')))
    o=ce(orig); return round(ce(proc)/o,4) if o else 1.0
def compute_CESS(I,B,E,a=0.33,b=0.33,g=0.34):
    return round(a*I+b*B+g*E,4)
def compute_NSR(orig,proc):
    NEG=list(NEGATION)
    def cn(texts): return sum(str(t).count(w) for t in texts for w in NEG)
    o=cn(orig); return round(cn(proc)/o,4) if o else 1.0

# ── BiLSTM ─────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self,input_dim=300,hidden=256,layers=2,dropout=0.3):
        super().__init__()
        self.lstm=nn.LSTM(input_dim,hidden,num_layers=layers,
            batch_first=True,bidirectional=True,
            dropout=dropout if layers>1 else 0.0)
        self.attn=nn.Linear(hidden*2,1)
        self.norm=nn.LayerNorm(hidden*2)
        self.drop=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden*2,2)
    def forward(self,x):
        out,_=self.lstm(x)
        w=torch.softmax(self.attn(out),dim=1)
        ctx=(w*out).sum(dim=1)
        return self.fc(self.drop(self.norm(ctx)))

def train_bilstm_seeded(X_train,y_train,X_val,y_val,
                        X_test,y_test,ft,seed,
                        epochs=20,batch_size=256,max_len=128):
    set_seed(seed)
    def emb(texts):
        all_e=[]
        for t in texts:
            toks=str(t).split()[:max_len]
            if not toks: toks=['unk']
            v=np.array([ft.get_word_vector(x) for x in toks],dtype=np.float32)
            while len(v)<max_len:
                v=np.vstack([v,np.zeros(300,dtype=np.float32)])
            all_e.append(v[:max_len])
        return torch.tensor(np.array(all_e),dtype=torch.float32)
    Xtr=emb(X_train); Xv=emb(X_val); Xte=emb(X_test)
    ytr=torch.tensor(y_train,dtype=torch.long)
    yv=torch.tensor(y_val,dtype=torch.long)
    yte=torch.tensor(y_test,dtype=torch.long)
    g=torch.Generator(); g.manual_seed(seed)
    tr_dl=DataLoader(TensorDataset(Xtr,ytr),batch_size=batch_size,
                     shuffle=True,num_workers=0,generator=g)
    v_dl=DataLoader(TensorDataset(Xv,yv),batch_size=batch_size*2)
    te_dl=DataLoader(TensorDataset(Xte,yte),batch_size=batch_size*2)
    model=BiLSTMClassifier().to(DEVICE)
    cnt=Counter(y_train); tot=len(y_train)
    wts=torch.tensor([tot/cnt[0],tot/cnt[1]],dtype=torch.float32).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
    crit=nn.CrossEntropyLoss(weight=wts)
    scaler=GradScaler()
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best_f1,best_state,patience=0.0,None,0
    for ep in range(epochs):
        model.train()
        for xb,yb in tr_dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast(): loss=crit(model(xb),yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update()
        sched.step()
        model.eval(); vp,vt=[],[]
        with torch.no_grad():
            for xb,yb in v_dl:
                vp.extend(model(xb.to(DEVICE)).argmax(1).cpu().tolist())
                vt.extend(yb.tolist())
        vf1=f1_score(vt,vp,average='weighted')
        if vf1>best_f1:
            best_f1=vf1; patience=0
            best_state={k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience+=1
            if patience>=5: break
    model.load_state_dict(best_state); model.eval()
    ap,ab,at=[],[],[]
    with torch.no_grad():
        for xb,yb in te_dl:
            lg=model(xb.to(DEVICE))
            ab.extend(torch.softmax(lg,1)[:,1].cpu().tolist())
            ap.extend(lg.argmax(1).cpu().tolist())
            at.extend(yb.tolist())
    del model; torch.cuda.empty_cache(); gc.collect()
    return at,ap,ab

# ── Transformer ─────────────────────────────────────────────
class NumpyDS(Dataset):
    def __init__(self,ids,mask,labels):
        self.ids=ids; self.mask=mask; self.labels=labels
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        return (torch.tensor(self.ids[i],dtype=torch.long),
                torch.tensor(self.mask[i],dtype=torch.long),
                torch.tensor(self.labels[i],dtype=torch.long))

def pretok(texts,labels,tok,max_len=128,chunk=1000):
    ids,mask=[],[]
    for i in range(0,len(texts),chunk):
        b=[str(t) for t in texts[i:i+chunk]]
        e=tok(b,truncation=True,padding='max_length',
              max_length=max_len,return_tensors='np')
        ids.append(e['input_ids'].astype(np.int32))
        mask.append(e['attention_mask'].astype(np.int32))
    return np.concatenate(ids,0),np.concatenate(mask,0),np.array(labels,dtype=np.int32)

def train_transformer_seeded(model_name,X_train,y_train,
                              X_val,y_val,X_test,y_test,
                              seed,epochs=8,batch_size=32,max_len=128):
    set_seed(seed)
    orig=mu.check_torch_load_is_safe
    mu.check_torch_load_is_safe=lambda:None
    tok=AutoTokenizer.from_pretrained(model_name)
    tr_i,tr_m,tr_l=pretok(X_train,y_train,tok,max_len)
    v_i,v_m,v_l  =pretok(X_val,  y_val,  tok,max_len)
    te_i,te_m,te_l=pretok(X_test, y_test, tok,max_len)
    del tok; gc.collect()
    g=torch.Generator(); g.manual_seed(seed)
    tr_dl=DataLoader(NumpyDS(tr_i,tr_m,tr_l),batch_size=batch_size,
                     shuffle=True,num_workers=0,generator=g)
    v_dl =DataLoader(NumpyDS(v_i,v_m,v_l),  batch_size=batch_size*2,num_workers=0)
    te_dl=DataLoader(NumpyDS(te_i,te_m,te_l),batch_size=batch_size*2,num_workers=0)
    model=AutoModelForSequenceClassification.from_pretrained(
        model_name,num_labels=2,ignore_mismatched_sizes=True).to(DEVICE)
    opt=AdamW(model.parameters(),lr=2e-5,weight_decay=0.01)
    scaler=GradScaler()
    total=len(tr_dl)*epochs
    sched=get_linear_schedule_with_warmup(opt,int(0.1*total),total)
    best_f1,best_state,patience=0.0,None,0
    for ep in range(epochs):
        model.train()
        for ids,mask,labs in tr_dl:
            ids,mask,labs=ids.to(DEVICE),mask.to(DEVICE),labs.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast():
                loss=model(input_ids=ids,attention_mask=mask,labels=labs).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step()
        model.eval(); vp,vt=[],[]
        with torch.no_grad():
            for ids,mask,labs in v_dl:
                vp.extend(model(input_ids=ids.to(DEVICE),
                    attention_mask=mask.to(DEVICE)).logits.argmax(1).cpu().tolist())
                vt.extend(labs.tolist())
        vf1=f1_score(vt,vp,average='weighted')
        if vf1>best_f1:
            best_f1=vf1; patience=0
            best_state={k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience+=1
            if patience>=3: break
    model.load_state_dict(best_state); model.eval()
    ap,ab,at=[],[],[]
    with torch.no_grad():
        for ids,mask,labs in te_dl:
            lg=model(input_ids=ids.to(DEVICE),
                     attention_mask=mask.to(DEVICE)).logits
            ab.extend(torch.softmax(lg,1)[:,1].cpu().tolist())
            ap.extend(lg.argmax(1).cpu().tolist())
            at.extend(labs.tolist())
    del model; torch.cuda.empty_cache(); gc.collect()
    mu.check_torch_load_is_safe=orig
    return at,ap,ab

# ── Verify ─────────────────────────────────────────────────
test_txt = "আমি আর পারছি না 😭 mon kharap নেই"
assert preprocess_P2(test_txt) != preprocess_P4_EPH(test_txt)
print("✅ ALL FUNCTIONS READY")
print("   preprocess_P0/P1/P2/P3/P4_EPH ✅")
print("   EPH_full/no_emoticon/no_negation/no_banglish/no_elongation ✅")
print("   compute_all_metrics/CESS/IPR/BSR/ELPS ✅")
print("   train_bilstm_seeded/train_transformer_seeded ✅")
print("   load_split/load_native_split/load_yt_split ✅")
print(f"\n   GPU: {torch.cuda.get_device_name(0)}")
print("   Run any experiment cell now.")

GPU: NVIDIA GeForce RTX 4080 SUPER
✅ ALL FUNCTIONS READY
   preprocess_P0/P1/P2/P3/P4_EPH ✅
   EPH_full/no_emoticon/no_negation/no_banglish/no_elongation ✅
   compute_all_metrics/CESS/IPR/BSR/ELPS ✅
   train_bilstm_seeded/train_transformer_seeded ✅
   load_split/load_native_split/load_yt_split ✅

   GPU: NVIDIA GeForce RTX 4080 SUPER
   Run any experiment cell now.


In [5]:
# CELL — Load + prepare YouTube annotated dataset

import pandas as pd
import re
from sklearn.model_selection import train_test_split

ann_path = './BanglishDetect/data/raw/annotation_batch_final.csv'

df = pd.read_csv(ann_path)

# Keep only manually labeled rows
df_ann = df[df['manual_label'].notna()].copy()

df_ann['label'] = df_ann['manual_label'].astype(int)
df_ann['source'] = 'youtube_native'

print(f"Total annotated: {len(df_ann)}")
print(f"Depressive:      {(df_ann['label']==1).sum()}")
print(f"Non-depressive:  {(df_ann['label']==0).sum()}")

# Agreement with tentative labels
if 'tentative_label' in df_ann.columns:
    agreed = (df_ann['label'] == df_ann['tentative_label']).sum()

    print(
        f"\nAgreement with tentative: "
        f"{agreed}/{len(df_ann)} "
        f"({100*agreed/len(df_ann):.1f}%)"
    )

# ── Emotion signal analysis ───────────────────────────────

texts = df_ann['text'].astype(str).tolist()

em = sum(
    any(e in t for e in EMOTICONS)
    for t in texts
)

bn = sum(
    bool(
        re.search(r'[a-zA-Z]{3,}', t)
        and re.search(r'[\u0980-\u09FF]', t)
    )
    for t in texts
)

el = sum(
    bool(re.search(r'(.)\1{2,}', t))
    for t in texts
)

neg = sum(
    any(w in t for w in NEGATION)
    for t in texts
)

n = len(texts)

print(f"\nEMOTION SIGNALS:")
print(f"  Emoticons:  {em}  ({100*em/n:.1f}%)")
print(f"  Banglish:   {bn}  ({100*bn/n:.1f}%)")
print(f"  Elongation: {el}  ({100*el/n:.1f}%)")
print(f"  Negation:   {neg} ({100*neg/n:.1f}%)")

# ── Balance dataset ───────────────────────────────────────

dep = df_ann[df_ann['label'] == 1]
non_dep = df_ann[df_ann['label'] == 0]

mn = min(len(dep), len(non_dep))

df_balanced = pd.concat([
    dep.sample(mn, random_state=42),
    non_dep.sample(mn, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset: {len(df_balanced)}")
print(f"({mn} depressive + {mn} non-depressive)")

# ── Train / validation / test split ──────────────────────

X = df_balanced['text']
y = df_balanced['label']

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_v, X_te, y_v, y_te = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    stratify=y_tmp,
    random_state=42
)

# ── Save splits ───────────────────────────────────────────

for name, Xs, ys in [
    ('yt_train', X_tr, y_tr),
    ('yt_val',   X_v,  y_v),
    ('yt_test',  X_te, y_te),
]:
    pd.DataFrame({
        'text': Xs,
        'label': ys
    }).to_csv(
        f"{base}{name}.csv",
        index=False
    )

print(f"\nSplits saved:")
print(f"  Train: {len(X_tr)}")
print(f"  Val:   {len(X_v)}")
print(f"  Test:  {len(X_te)}")

print("\n✅ Ready for experiments")

Total annotated: 1000
Depressive:      417
Non-depressive:  583

Agreement with tentative: 680/1000 (68.0%)

EMOTION SIGNALS:
  Emoticons:  541  (54.1%)
  Banglish:   544  (54.4%)
  Elongation: 411  (41.1%)
  Negation:   678 (67.8%)

Balanced dataset: 834
(417 depressive + 417 non-depressive)

Splits saved:
  Train: 583
  Val:   125
  Test:  126

✅ Ready for experiments


In [6]:
# CELL — Apply all 5 preprocessing conditions to YouTube data
# This is the KEY experiment — will show meaningful CESS differences

import pandas as pd
import os

proc_base = './BanglishDetect/data/processed/'
yt_base   = './BanglishDetect/data/raw/'

# Preprocessing functions must be defined
# Run master setup cell first if kernel restarted

preprocessing = {
    'P0': preprocess_P0,
    'P1': preprocess_P1,
    'P2': preprocess_P2,
    'P3': preprocess_P3,
    'P4': preprocess_P4_EPH,
}

splits = {
    'train': pd.read_csv(yt_base + 'yt_train.csv'),
    'val':   pd.read_csv(yt_base + 'yt_val.csv'),
    'test':  pd.read_csv(yt_base + 'yt_test.csv'),
}

print("Applying preprocessing to YouTube corpus...")
for cond, func in preprocessing.items():
    for split_name, df in splits.items():
        df_proc = df.copy()
        df_proc['text'] = df_proc['text'].apply(func)
        df_proc = df_proc[
            df_proc['text'].str.strip().str.len() > 0
        ]
        out = f"{proc_base}yt_{cond}_{split_name}.csv"
        df_proc.to_csv(out, index=False)
        print(f"  yt_{cond}_{split_name}: {len(df_proc)} rows")

print("\n✅ YouTube preprocessing complete")

Applying preprocessing to YouTube corpus...
  yt_P0_train: 583 rows
  yt_P0_val: 125 rows
  yt_P0_test: 126 rows
  yt_P1_train: 583 rows
  yt_P1_val: 125 rows
  yt_P1_test: 126 rows
  yt_P2_train: 583 rows
  yt_P2_val: 125 rows
  yt_P2_test: 126 rows
  yt_P3_train: 583 rows
  yt_P3_val: 125 rows
  yt_P3_test: 126 rows
  yt_P4_train: 583 rows
  yt_P4_val: 125 rows
  yt_P4_test: 126 rows

✅ YouTube preprocessing complete


In [7]:
# CELL — Compute CESS on YouTube corpus
# THIS is where CESS will show real differences
# Unlike BSMDD where everything was 0.9998

import math
from collections import Counter

yt_train = pd.read_csv('./BanglishDetect/data/raw/yt_train.csv')
yt_texts = yt_train['text'].astype(str).tolist()

print("CESS ON YOUTUBE NATIVE CORPUS")
print("="*55)
print(f"Corpus size: {len(yt_texts)} samples\n")

funcs = {
    'P0 (Raw)':       preprocess_P0,
    'P1 (Minimal)':   preprocess_P1,
    'P2 (Standard)':  preprocess_P2,
    'P3 (Aggressive)':preprocess_P3,
    'P4 (EPH)':       preprocess_P4_EPH,
}

cess_results = {}
print(f"{'Condition':<20} {'IPR':>6} {'BSR':>6} "
      f"{'ELPS':>6} {'CESS':>6}")
print("-"*50)

for name, func in funcs.items():
    proc = [func(t) for t in yt_texts]
    ipr  = compute_IPR(yt_texts, proc)
    bsr  = compute_BSR(yt_texts, proc)
    elps = compute_ELPS(yt_texts, proc)
    cess = compute_CESS(ipr, bsr, elps)
    cess_results[name] = {
        'IPR':ipr,'BSR':bsr,'ELPS':elps,'CESS':cess}
    print(f"{name:<20} {ipr:>6} {bsr:>6} {elps:>6} {cess:>6}")

print("\n✅ This table = Table 3 in paper")
print("P2/P3 should show BSR drop (destroys Banglish)")
print("P4 should maintain high BSR (preserves Banglish)")

CESS ON YOUTUBE NATIVE CORPUS
Corpus size: 583 samples

Condition               IPR    BSR   ELPS   CESS
--------------------------------------------------
P0 (Raw)                1.0    1.0    1.0    1.0
P1 (Minimal)         0.9996 0.9942    1.0  0.998
P2 (Standard)        0.9147    0.0 1.0033  0.643
P3 (Aggressive)      0.9124    0.0 1.0033 0.6422
P4 (EPH)             0.9683 1.4048 4.9331 2.4604

✅ This table = Table 3 in paper
P2/P3 should show BSR drop (destroys Banglish)
P4 should maintain high BSR (preserves Banglish)


In [8]:
# CELL — Run all 4 models on YouTube corpus (3 seeds)
# Same structure as BSMDD experiments

import torch, gc, fasttext
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

YT_RESULTS = []

def load_yt_split(condition, split):
    path = f"./BanglishDetect/data/processed/yt_{condition}_{split}.csv"
    df   = pd.read_csv(path).dropna(subset=['text'])
    df['text'] = df['text'].astype(str)
    return df['text'].tolist(), df['label'].tolist()

def log_yt(model, condition, seed, metrics):
    YT_RESULTS.append({
        'model':model,'corpus':'youtube',
        'condition':condition,'seed':seed,**metrics
    })
    print(f"    {condition} seed={seed} → "
          f"F1={metrics['f1']} MCC={metrics['mcc']}")

conditions = ['P0','P1','P2','P3','P4']
SEEDS      = [42, 123, 456]

# ── SVM ───────────────────────────────────────────────────
print("MODEL 1: SVM — YouTube corpus")
print("="*45)
p0_svm = {}
for cond in conditions:
    f1s = []
    for seed in SEEDS:
        set_seed(seed)
        X_tr,y_tr = load_yt_split(cond,'train')
        X_te,y_te = load_yt_split(cond,'test')
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(
                ngram_range=(1,2), max_features=30000,
                sublinear_tf=True, min_df=1)),
            ('clf',  CalibratedClassifierCV(
                LinearSVC(C=1.0,random_state=seed,
                          max_iter=2000), cv=3))
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        y_prob = pipe.predict_proba(X_te)[:,1]
        m = compute_all_metrics(y_te, y_pred, y_prob)
        log_yt('SVM', cond, seed, m)
        f1s.append(m['f1'])
    if cond == 'P0': p0_svm['yt'] = np.mean(f1s)
    print(f"  {cond}: F1={np.mean(f1s):.4f}±{np.std(f1s):.4f}")

# ── BiLSTM ────────────────────────────────────────────────
print("\nMODEL 2: BiLSTM — YouTube corpus")
print("="*45)
print("Loading FastText...")
ft = fasttext.load_model('./cc.bn.300.bin')
print("✅")

for cond in conditions:
    f1s = []
    for seed in SEEDS:
        set_seed(seed)
        X_tr,y_tr = load_yt_split(cond,'train')
        X_v, y_v  = load_yt_split(cond,'val')
        X_te,y_te = load_yt_split(cond,'test')
        y_true,y_pred,y_prob = train_bilstm_seeded(
            X_tr,y_tr,X_v,y_v,X_te,y_te,
            ft,seed=seed,epochs=20,batch_size=128)
        m = compute_all_metrics(y_true,y_pred,y_prob)
        log_yt('BiLSTM', cond, seed, m)
        f1s.append(m['f1'])
    print(f"  {cond}: F1={np.mean(f1s):.4f}±{np.std(f1s):.4f}")

del ft; gc.collect()

# ── BanglaBERT ────────────────────────────────────────────
print("\nMODEL 3: BanglaBERT — YouTube corpus")
print("="*45)
for cond in conditions:
    f1s = []
    for seed in SEEDS:
        set_seed(seed)
        X_tr,y_tr = load_yt_split(cond,'train')
        X_v, y_v  = load_yt_split(cond,'val')
        X_te,y_te = load_yt_split(cond,'test')
        y_true,y_pred,y_prob = train_transformer_seeded(
            'sagorsarker/bangla-bert-base',
            X_tr,y_tr,X_v,y_v,X_te,y_te,
            seed=seed,epochs=8,batch_size=32)
        m = compute_all_metrics(y_true,y_pred,y_prob)
        log_yt('BanglaBERT', cond, seed, m)
        f1s.append(m['f1'])
    print(f"  {cond}: F1={np.mean(f1s):.4f}±{np.std(f1s):.4f}")

# ── SahajBERT ─────────────────────────────────────────────
print("\nMODEL 4: SahajBERT — YouTube corpus")
print("="*45)
for cond in conditions:
    f1s = []
    for seed in SEEDS:
        set_seed(seed)
        X_tr,y_tr = load_yt_split(cond,'train')
        X_v, y_v  = load_yt_split(cond,'val')
        X_te,y_te = load_yt_split(cond,'test')
        y_true,y_pred,y_prob = train_transformer_seeded(
            'neuropark/sahajBERT',
            X_tr,y_tr,X_v,y_v,X_te,y_te,
            seed=seed,epochs=8,batch_size=32)
        m = compute_all_metrics(y_true,y_pred,y_prob)
        log_yt('SahajBERT', cond, seed, m)
        f1s.append(m['f1'])
    print(f"  {cond}: F1={np.mean(f1s):.4f}±{np.std(f1s):.4f}")

# Save
import pandas as pd
df_yt = pd.DataFrame(YT_RESULTS)
df_yt.to_csv('./BanglishDetect/results/youtube_results.csv',
             index=False)
print(f"\n✅ Saved {len(df_yt)} YouTube experiment rows")

# Quick EPH check
print("\nEPH (P4) vs P2 on YouTube:")
print("="*45)
agg = df_yt.groupby(['model','condition'])['f1'].mean()
for m in ['SVM','BiLSTM','BanglaBERT','SahajBERT']:
    try:
        p2 = agg[m]['P2']
        p4 = agg[m]['P4']
        d  = round(p4-p2, 4)
        flag = '✅ EPH wins' if d>0 else '❌ P2 wins'
        print(f"  {m:<14}: P2={p2:.4f} P4={p4:.4f} "
              f"diff={d:+.4f} {flag}")
    except: pass

MODEL 1: SVM — YouTube corpus
    P0 seed=42 → F1=0.7458 MCC=0.4931
    P0 seed=123 → F1=0.7458 MCC=0.4931
    P0 seed=456 → F1=0.7458 MCC=0.4931
  P0: F1=0.7458±0.0000
    P1 seed=42 → F1=0.7301 MCC=0.4605
    P1 seed=123 → F1=0.7301 MCC=0.4605
    P1 seed=456 → F1=0.7301 MCC=0.4605
  P1: F1=0.7301±0.0000
    P2 seed=42 → F1=0.7062 MCC=0.4132
    P2 seed=123 → F1=0.7062 MCC=0.4132
    P2 seed=456 → F1=0.7062 MCC=0.4132
  P2: F1=0.7062±0.0000
    P3 seed=42 → F1=0.7063 MCC=0.4128
    P3 seed=123 → F1=0.7063 MCC=0.4128
    P3 seed=456 → F1=0.7063 MCC=0.4128
  P3: F1=0.7063±0.0000
    P4 seed=42 → F1=0.857 MCC=0.7157
    P4 seed=123 → F1=0.857 MCC=0.7157
    P4 seed=456 → F1=0.857 MCC=0.7157
  P4: F1=0.8570±0.0000

MODEL 2: BiLSTM — YouTube corpus
Loading FastText...
✅
    P0 seed=42 → F1=0.3333 MCC=0.0
    P0 seed=123 → F1=0.7764 MCC=0.5627
    P0 seed=456 → F1=0.8247 MCC=0.6561
  P0: F1=0.6448±0.2211
    P1 seed=42 → F1=0.3333 MCC=0.0
    P1 seed=123 → F1=0.7826 MCC=0.5883
    P1 seed=

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    P0 seed=42 → F1=0.857 MCC=0.7157


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    P0 seed=123 → F1=0.849 MCC=0.7006


...

    P4 seed=42 → F1=0.8333 MCC=0.6668


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

    P4 seed=123 → F1=0.8491 MCC=0.6992


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

    P4 seed=456 → F1=0.8403 MCC=0.6913
  P4: F1=0.8409±0.0065

✅ Saved 60 YouTube experiment rows

EPH (P4) vs P2 on YouTube:
  SVM           : P2=0.7062 P4=0.8570 diff=+0.1508 ✅ EPH wins
  BiLSTM        : P2=0.6737 P4=0.6374 diff=-0.0364 ❌ P2 wins
  BanglaBERT    : P2=0.8355 P4=0.8621 diff=+0.0266 ✅ EPH wins
  SahajBERT     : P2=0.8411 P4=0.8409 diff=-0.0002 ❌ P2 wins


In [2]:
# CELL — Use best model to pseudo-label 191k
# Then use pseudo-labels as additional training data
# This is standard NLP practice, well-accepted by reviewers

import pandas as pd
import numpy as np
import torch
import gc

# Load the 191k clean comments
df_full = pd.read_csv(
    './BanglishDetect/data/raw/youtube_collected_raw.csv')

# Apply EPH preprocessing (best condition)
print("Applying EPH to 191k comments...")
df_full['text_eph'] = df_full['text'].astype(str).apply(
    preprocess_P4_EPH)

# Remove empty
df_full = df_full[
    df_full['text_eph'].str.strip().str.len() > 0
].reset_index(drop=True)
print(f"After EPH: {len(df_full):,} comments")

# Use SahajBERT (best model) trained on 1000 labeled
# to predict probability on all 191k

# Load the model trained on YouTube 1000
# (retrain quickly if needed)
SAHAJBERT = 'neuropark/sahajBERT'

# Train on full 1000 labeled first
X_tr, y_tr = load_yt_split('P4','train')
X_v,  y_v  = load_yt_split('P4','val')
X_te, y_te = load_yt_split('P4','test')

print("Training SahajBERT on 1000 labeled...")
set_seed(42)
y_true, y_pred, y_prob = train_transformer_seeded(
    SAHAJBERT, X_tr, y_tr, X_v, y_v, X_te, y_te,
    seed=42, epochs=8, batch_size=32
)
m = compute_all_metrics(y_true, y_pred, y_prob)
print(f"Model quality: F1={m['f1']} MCC={m['mcc']}")
print("Use this model to predict on 191k")

Applying EPH to 191k comments...
After EPH: 190,992 comments
Training SahajBERT on 1000 labeled...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

Model quality: F1=0.8333 MCC=0.6668
Use this model to predict on 191k


In [3]:
# CELL — Pseudo-label 191k with confidence threshold
# Only keep HIGH CONFIDENCE predictions
# This is called "confidence-based pseudo-labeling"
# Standard technique — ACL/EMNLP accept this

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

DEVICE    = torch.device('cuda:0')
SAHAJBERT = 'neuropark/sahajBERT'
THRESHOLD = 0.85  # only keep if model is 85%+ confident

# Load 191k EPH-processed texts
texts_191k = df_full['text_eph'].astype(str).tolist()
print(f"Predicting on {len(texts_191k):,} comments...")

# Re-load best trained model
# (use the one trained above — still in memory if not deleted)
import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

tok   = AutoTokenizer.from_pretrained(SAHAJBERT)
model = AutoModelForSequenceClassification.from_pretrained(
    SAHAJBERT, num_labels=2,
    ignore_mismatched_sizes=True
).to(DEVICE)

# Load best weights if saved
# model.load_state_dict(best_state)  # if still in memory
model.eval()

# Predict in batches
BATCH = 64
all_probs  = []
all_labels = []

print("Predicting...")
for i in range(0, len(texts_191k), BATCH):
    batch_texts = texts_191k[i:i+BATCH]
    enc = tok(batch_texts, truncation=True,
               padding=True, max_length=128,
               return_tensors='pt')

    with torch.no_grad():
        logits = model(
            input_ids      = enc['input_ids'].to(DEVICE),
            attention_mask = enc['attention_mask'].to(DEVICE)
        ).logits
        probs  = torch.softmax(logits, 1)
        conf,  pred = probs.max(dim=1)

    all_probs.extend(conf.cpu().tolist())
    all_labels.extend(pred.cpu().tolist())

    if i % 10000 == 0:
        print(f"  {i:,}/{len(texts_191k):,}")

del model, tok
torch.cuda.empty_cache()
mu.check_torch_load_is_safe = lambda: None

# Add predictions to dataframe
df_full['pseudo_label']      = all_labels
df_full['pseudo_confidence'] = all_probs

# ── Keep only high-confidence predictions ─────────────────
df_high_conf = df_full[
    df_full['pseudo_confidence'] >= THRESHOLD
].copy()

print(f"\nTotal comments:          {len(df_full):,}")
print(f"High confidence (≥{THRESHOLD}): {len(df_high_conf):,} "
      f"({100*len(df_high_conf)/len(df_full):.1f}%)")
print(f"\nPseudo-label distribution:")
print(df_high_conf['pseudo_label'].value_counts().to_dict())

# Save pseudo-labeled dataset
df_high_conf[['text_eph','pseudo_label',
              'pseudo_confidence','category']].to_csv(
    './BanglishDetect/data/raw/pseudo_labeled_191k.csv',
    index=False)
print(f"\n✅ Pseudo-labeled dataset saved")

Predicting on 190,992 comments...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

Predicting...
  0/190,992
  40,000/190,992
  80,000/190,992
  120,000/190,992
  160,000/190,992

Total comments:          190,992
High confidence (≥0.85): 214 (0.1%)

Pseudo-label distribution:
{0: 214}

✅ Pseudo-labeled dataset saved


In [4]:
# CELL — Train models on combined dataset
# 1000 manual labels + pseudo-labeled high-confidence
# Compare performance vs 1000-only

import pandas as pd
import numpy as np

# Load manually labeled
df_manual = pd.read_csv(
    './BanglishDetect/data/raw/annotation_batch_final.csv')
df_manual = df_manual[df_manual['manual_label'].notna()].copy()
df_manual['label'] = df_manual['manual_label'].astype(int)
df_manual['text']  = df_manual['text'].apply(preprocess_P4_EPH)

# Load pseudo-labeled
df_pseudo = pd.read_csv(
    './BanglishDetect/data/raw/pseudo_labeled_191k.csv')
df_pseudo = df_pseudo.rename(
    columns={'text_eph':'text','pseudo_label':'label'})

# Combine
df_combined = pd.concat([
    df_manual[['text','label']],
    df_pseudo[['text','label']],
], ignore_index=True)

# Balance
dep     = df_combined[df_combined['label']==1]
non_dep = df_combined[df_combined['label']==0]
mn      = min(len(dep), len(non_dep))
df_bal  = pd.concat([
    dep.sample(mn, random_state=42),
    non_dep.sample(mn, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Manual only:       {len(df_manual):,}")
print(f"Pseudo-labeled:    {len(df_pseudo):,}")
print(f"Combined balanced: {len(df_bal):,}")

# Split + save
from sklearn.model_selection import train_test_split

X,y = df_bal['text'], df_bal['label']
X_tr,X_tmp,y_tr,y_tmp = train_test_split(
    X,y,test_size=0.30,stratify=y,random_state=42)
X_v,X_te,y_v,y_te = train_test_split(
    X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=42)

proc_base = './BanglishDetect/data/processed/'
for name,Xs,ys in [
    ('combined_train',X_tr,y_tr),
    ('combined_val',  X_v, y_v),
    ('combined_test', X_te,y_te),
]:
    pd.DataFrame({'text':Xs,'label':ys}).to_csv(
        f"{proc_base}{name}.csv", index=False)

print(f"\nSplit: Train={len(X_tr)} Val={len(X_v)} Test={len(X_te)}")
print("✅ Combined dataset ready")
print("\nNow train SahajBERT on combined vs 1000-only")
print("If combined > 1000-only → pseudo-labeling helps")
print("Report both in paper as ablation")

Manual only:       1,000
Pseudo-labeled:    214
Combined balanced: 834

Split: Train=583 Val=125 Test=126
✅ Combined dataset ready

Now train SahajBERT on combined vs 1000-only
If combined > 1000-only → pseudo-labeling helps
Report both in paper as ablation


In [1]:
#ChatGPT

In [11]:
# ============================================================
# CELL 1 — TRAIN BEST MODEL ON 1000 HUMAN-LABELED DATA
# ============================================================
# IMPORTANT:
# - This creates the ONLY official test set
# - Test set must NEVER change afterward
# - Pseudo-labeled samples NEVER enter test set
# ============================================================

import os
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

SAHAJBERT = 'neuropark/sahajBERT'

# ============================================================
# LOAD HUMAN-LABELED DATA
# ============================================================

df_manual = pd.read_csv(
    './BanglishDetect/data/raw/annotation_batch_final.csv'
)

df_manual = df_manual[
    df_manual['manual_label'].notna()
].copy()

df_manual['label'] = df_manual['manual_label'].astype(int)

# Apply EPH
df_manual['text'] = df_manual['text'].astype(str).apply(
    preprocess_P4_EPH
)

df_manual = df_manual[
    df_manual['text'].str.strip().str.len() > 0
].reset_index(drop=True)

print(f"Human-labeled samples: {len(df_manual)}")

# ============================================================
# CREATE FROZEN TRAIN / VAL / TEST SPLIT
# ============================================================

X = df_manual['text']
y = df_manual['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print(f"\nTrain: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

# ============================================================
# SAVE FROZEN SPLITS
# ============================================================

SAVE_DIR = './BanglishDetect/data/processed/'

pd.DataFrame({
    'text': X_train,
    'label': y_train
}).to_csv(f'{SAVE_DIR}gold_train.csv', index=False)

pd.DataFrame({
    'text': X_val,
    'label': y_val
}).to_csv(f'{SAVE_DIR}gold_val.csv', index=False)

pd.DataFrame({
    'text': X_test,
    'label': y_test
}).to_csv(f'{SAVE_DIR}gold_test.csv', index=False)

print("\n✅ Frozen gold splits saved")

Human-labeled samples: 1000

Train: 700
Val:   150
Test:  150

✅ Frozen gold splits saved


In [16]:
# ============================================================
# CELL 2 — TRAIN SAHAJBERT ON HUMAN-LABELED TRAIN SET
# ============================================================

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef
)
import transformers.modeling_utils as mu

mu.check_torch_load_is_safe = lambda: None

tokenizer = AutoTokenizer.from_pretrained(SAHAJBERT)

train_df = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

val_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

test_df = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

cols = ['input_ids', 'attention_mask', 'label']

train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

model = AutoModelForSequenceClassification.from_pretrained(
    SAHAJBERT,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'mcc': matthews_corrcoef(labels, preds)
    }

training_args = TrainingArguments(
    output_dir='./sahajbert_gold_model',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print("\nTraining SahajBERT...")
trainer.train()

print("\nEvaluating on GOLD TEST SET...")
gold_results = trainer.evaluate(test_ds)

print("\nGold-only Results:")
for k, v in gold_results.items():
    print(f"{k}: {v}")

# SAVE BEST MODEL
trainer.save_model('./best_sahajbert_gold')
tokenizer.save_pretrained('./best_sahajbert_gold')

print("\n✅ Best model saved")

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]


Training SahajBERT...
{'loss': '0.5917', 'grad_norm': '9.814', 'learning_rate': '1.761e-05', 'epoch': '1'}
{'eval_loss': '0.5157', 'eval_accuracy': '0.7867', 'eval_f1': '0.6667', 'eval_precision': '0.9697', 'eval_recall': '0.5079', 'eval_mcc': '0.5915', 'eval_runtime': '0.6564', 'eval_samples_per_second': '228.5', 'eval_steps_per_second': '7.617', 'epoch': '1'}


...

{'loss': '0.01622', 'grad_norm': '0.4241', 'learning_rate': '1.136e-07', 'epoch': '8'}
{'eval_loss': '0.7501', 'eval_accuracy': '0.82', 'eval_f1': '0.7769', 'eval_precision': '0.8103', 'eval_recall': '0.746', 'eval_mcc': '0.628', 'eval_runtime': '0.662', 'eval_samples_per_second': '226.6', 'eval_steps_per_second': '7.553', 'epoch': '8'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '73.09', 'train_samples_per_second': '76.62', 'train_steps_per_second': '2.408', 'train_loss': '0.1992', 'epoch': '8'}

Evaluating on GOLD TEST SET...
{'eval_loss': '0.8803', 'eval_accuracy': '0.7933', 'eval_f1': '0.7438', 'eval_precision': '0.7627', 'eval_recall': '0.7258', 'eval_mcc': '0.5713', 'eval_runtime': '0.651', 'eval_samples_per_second': '230.4', 'eval_steps_per_second': '7.68', 'epoch': '8'}

Gold-only Results:
eval_loss: 0.8802558183670044
eval_accuracy: 0.7933333333333333
eval_f1: 0.743801652892562
eval_precision: 0.7627118644067796
eval_recall: 0.7258064516129032
eval_mcc: 0.571288658657033
eval_runtime: 0.651
eval_samples_per_second: 230.398
eval_steps_per_second: 7.68
epoch: 8.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Best model saved


In [17]:
# ============================================================
# CELL 3 — LOAD 191K UNLABELED DATA + APPLY EPH
# ============================================================

df_full = pd.read_csv(
    './BanglishDetect/data/raw/youtube_collected_raw.csv'
)

print(f"Raw comments: {len(df_full):,}")

print("\nApplying EPH preprocessing...")

df_full['text_eph'] = df_full['text'].astype(str).apply(
    preprocess_P4_EPH
)

df_full = df_full[
    df_full['text_eph'].str.strip().str.len() > 0
].reset_index(drop=True)

print(f"After EPH: {len(df_full):,}")

Raw comments: 191,000

Applying EPH preprocessing...
After EPH: 190,992


In [18]:
# ============================================================
# CELL 4 — PSEUDO-LABEL 191K USING TRAINED MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import torch.nn.functional as F

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

THRESHOLD = 0.90

texts_191k = df_full['text_eph'].astype(str).tolist()

print(f"Predicting on {len(texts_191k):,} comments")

# ============================================================
# LOAD TRAINED MODEL (IMPORTANT)
# ============================================================

tok = AutoTokenizer.from_pretrained(
    './best_sahajbert_gold'
)

model = AutoModelForSequenceClassification.from_pretrained(
    './best_sahajbert_gold'
).to(DEVICE)

model.eval()

all_probs  = []
all_labels = []

BATCH = 64

for i in tqdm(range(0, len(texts_191k), BATCH)):

    batch_texts = texts_191k[i:i+BATCH]

    enc = tok(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors='pt'
    )

    enc = {k:v.to(DEVICE) for k,v in enc.items()}

    with torch.no_grad():

        outputs = model(**enc)

        probs = F.softmax(outputs.logits, dim=1)

        conf, pred = probs.max(dim=1)

    all_probs.extend(conf.cpu().numpy())
    all_labels.extend(pred.cpu().numpy())

df_full['pseudo_label'] = all_labels
df_full['pseudo_confidence'] = all_probs

# ============================================================
# KEEP ONLY HIGH-CONFIDENCE SAMPLES
# ============================================================

df_pseudo = df_full[
    df_full['pseudo_confidence'] >= THRESHOLD
].copy()

print(f"\nTotal comments: {len(df_full):,}")
print(f"High-confidence comments: {len(df_pseudo):,}")

print("\nPseudo-label distribution:")
print(df_pseudo['pseudo_label'].value_counts())

# SAVE
df_pseudo[
    ['text_eph', 'pseudo_label', 'pseudo_confidence']
].to_csv(
    './BanglishDetect/data/raw/pseudo_labeled_highconf.csv',
    index=False
)

print("\n✅ Pseudo-labeled dataset saved")

Predicting on 190,992 comments


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████| 2985/2985 [10:23<00:00,  4.79it/s]



Total comments: 190,992
High-confidence comments: 172,629

Pseudo-label distribution:
pseudo_label
0    143096
1     29533
Name: count, dtype: int64

✅ Pseudo-labeled dataset saved


In [19]:
# ============================================================
# CELL 5 — CREATE COMBINED TRAINING SET
# ============================================================
# IMPORTANT:
# - ONLY TRAINING SET GETS PSEUDO-LABELS
# - VALIDATION + TEST REMAIN HUMAN-LABELED ONLY
# ============================================================

df_pseudo = pd.read_csv(
    './BanglishDetect/data/raw/pseudo_labeled_highconf.csv'
)

df_pseudo = df_pseudo.rename(columns={
    'text_eph': 'text',
    'pseudo_label': 'label'
})

gold_train_df = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

gold_val_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

gold_test_df = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

# ============================================================
# COMBINE ONLY TRAINING DATA
# ============================================================

combined_train_df = pd.concat([
    gold_train_df,
    df_pseudo[['text', 'label']]
], ignore_index=True)

print(f"Gold train:      {len(gold_train_df):,}")
print(f"Pseudo-labeled:  {len(df_pseudo):,}")
print(f"Combined train:  {len(combined_train_df):,}")

# SAVE
combined_train_df.to_csv(
    './BanglishDetect/data/processed/combined_train.csv',
    index=False
)

gold_val_df.to_csv(
    './BanglishDetect/data/processed/combined_val.csv',
    index=False
)

gold_test_df.to_csv(
    './BanglishDetect/data/processed/combined_test.csv',
    index=False
)

print("\n✅ Combined dataset saved")

Gold train:      700
Pseudo-labeled:  172,629
Combined train:  173,329

✅ Combined dataset saved


In [21]:
# ============================================================
# CELL 6 — RETRAIN WITH PSEUDO-LABELED DATA
# ============================================================

combined_train_df = pd.read_csv(
    './BanglishDetect/data/processed/combined_train.csv'
)

combined_train_ds = Dataset.from_pandas(
    combined_train_df
)

combined_train_ds = combined_train_ds.map(
    tokenize,
    batched=True
)

combined_train_ds.set_format(
    type='torch',
    columns=cols
)

# Reload fresh model
model_combined = AutoModelForSequenceClassification.from_pretrained(
    SAHAJBERT,
    num_labels=2
)

training_args_combined = TrainingArguments(
    output_dir='./sahajbert_combined_model',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    seed=SEED
)

trainer_combined = Trainer(
    model=model_combined,
    args=training_args_combined,
    train_dataset=combined_train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print("\nTraining with pseudo-labeled data...")
trainer_combined.train()

print("\nEvaluating on ORIGINAL GOLD TEST SET...")
combined_results = trainer_combined.evaluate(test_ds)

print("\nCombined Results:")
for k, v in combined_results.items():
    print(f"{k}: {v}")

print("\n✅ Semi-supervised experiment complete")

Map:   0%|          | 0/173329 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]


Training with pseudo-labeled data...
{'loss': '0.1008', 'grad_norm': '0.3495', 'learning_rate': '1.6e-05', 'epoch': '1'}
{'eval_loss': '0.8466', 'eval_accuracy': '0.8467', 'eval_f1': '0.8067', 'eval_precision': '0.8571', 'eval_recall': '0.7619', 'eval_mcc': '0.6836', 'eval_runtime': '0.6569', 'eval_samples_per_second': '228.3', 'eval_steps_per_second': '7.611', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.06348', 'grad_norm': '50.85', 'learning_rate': '1.2e-05', 'epoch': '2'}
{'eval_loss': '0.8615', 'eval_accuracy': '0.84', 'eval_f1': '0.8065', 'eval_precision': '0.8197', 'eval_recall': '0.7937', 'eval_mcc': '0.6704', 'eval_runtime': '0.6648', 'eval_samples_per_second': '225.6', 'eval_steps_per_second': '7.521', 'epoch': '2'}


...

{'loss': '0.01475', 'grad_norm': '10.56', 'learning_rate': '4.001e-06', 'epoch': '4'}
{'eval_loss': '1.151', 'eval_accuracy': '0.8467', 'eval_f1': '0.813', 'eval_precision': '0.8333', 'eval_recall': '0.7937', 'eval_mcc': '0.6838', 'eval_runtime': '0.6572', 'eval_samples_per_second': '228.2', 'eval_steps_per_second': '7.608', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.0034', 'grad_norm': '0.0004278', 'learning_rate': '7.384e-10', 'epoch': '5'}
{'eval_loss': '1.734', 'eval_accuracy': '0.8333', 'eval_f1': '0.7934', 'eval_precision': '0.8276', 'eval_recall': '0.7619', 'eval_mcc': '0.6557', 'eval_runtime': '0.6775', 'eval_samples_per_second': '221.4', 'eval_steps_per_second': '7.38', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1.031e+04', 'train_samples_per_second': '84.04', 'train_steps_per_second': '2.626', 'train_loss': '0.04372', 'epoch': '5'}

Evaluating on ORIGINAL GOLD TEST SET...
{'eval_loss': '1.627', 'eval_accuracy': '0.7933', 'eval_f1': '0.7438', 'eval_precision': '0.7627', 'eval_recall': '0.7258', 'eval_mcc': '0.5713', 'eval_runtime': '0.637', 'eval_samples_per_second': '235.5', 'eval_steps_per_second': '7.849', 'epoch': '5'}

Combined Results:
eval_loss: 1.6266961097717285
eval_accuracy: 0.7933333333333333
eval_f1: 0.743801652892562
eval_precision: 0.7627118644067796
eval_recall: 0.7258064516129032
eval_mcc: 0.571288658657033
eval_runtime: 0.637
eval_samples_per_second: 235.468
eval_steps_per_second: 7.849
epoch: 5.0

✅ Semi-supervised experiment complete


In [22]:
"""Try new idea"""

'Try new idea'

In [23]:
# ============================================================
# CELL 1 — TRAIN BEST MODEL ON 1000 HUMAN-LABELED DATA
# ============================================================
# IMPROVED VERSION:
# - stable seeds
# - early stopping
# - better hyperparameters
# - fp16 training
# - safer CUDA
# ============================================================

import os
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

SEED = 42

os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(SEED)

DEVICE = torch.device(
    'cuda:0' if torch.cuda.is_available() else 'cpu'
)

print("DEVICE:", DEVICE)

SAHAJBERT = 'neuropark/sahajBERT'

# ============================================================
# LOAD HUMAN-LABELED DATA
# ============================================================

df_manual = pd.read_csv(
    './BanglishDetect/data/raw/annotation_batch_final.csv'
)

df_manual = df_manual[
    df_manual['manual_label'].notna()
].copy()

df_manual['label'] = (
    df_manual['manual_label'].astype(int)
)

# ============================================================
# APPLY EPH
# ============================================================

print("\nApplying EPH preprocessing...")

texts = (
    df_manual['text']
    .fillna('')
    .astype(str)
    .str[:1000]
    .tolist()
)

processed = []

for i, txt in enumerate(texts):

    if i % 100 == 0:
        print(i)

    try:
        out = preprocess_P4_EPH(txt)

    except:
        out = ""

    processed.append(out)

df_manual['text'] = processed

df_manual = df_manual[
    df_manual['text'].str.strip().str.len() > 0
].reset_index(drop=True)

print(f"\nHuman-labeled samples: {len(df_manual)}")

# ============================================================
# CREATE FROZEN TRAIN / VAL / TEST SPLIT
# ============================================================

X = df_manual['text']
y = df_manual['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print(f"\nTrain: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

# ============================================================
# SAVE SPLITS
# ============================================================

SAVE_DIR = './BanglishDetect/data/processed/'

pd.DataFrame({
    'text': X_train,
    'label': y_train
}).to_csv(f'{SAVE_DIR}gold_train.csv', index=False)

pd.DataFrame({
    'text': X_val,
    'label': y_val
}).to_csv(f'{SAVE_DIR}gold_val.csv', index=False)

pd.DataFrame({
    'text': X_test,
    'label': y_test
}).to_csv(f'{SAVE_DIR}gold_test.csv', index=False)

print("\n✅ Frozen gold splits saved")

DEVICE: cuda:0

Applying EPH preprocessing...
0
100
200
300
400
500
600
700
800
900

Human-labeled samples: 1000

Train: 700
Val:   150
Test:  150

✅ Frozen gold splits saved


In [24]:
# ============================================================
# CELL 2 — TRAIN SAHAJBERT ON GOLD DATA
# ============================================================

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef
)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    SAHAJBERT
)

# ============================================================
# DATAFRAMES
# ============================================================

train_df = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

val_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

test_df = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

# ============================================================
# TOKENIZATION
# ============================================================

def tokenize(batch):

    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

cols = [
    'input_ids',
    'attention_mask',
    'label'
]

train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

# ============================================================
# MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    SAHAJBERT,
    num_labels=2
)

model.to(DEVICE)

# ============================================================
# METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {

        'accuracy':
            accuracy_score(labels, preds),

        'f1':
            f1_score(labels, preds),

        'precision':
            precision_score(labels, preds),

        'recall':
            recall_score(labels, preds),

        'mcc':
            matthews_corrcoef(labels, preds)
    }

# ============================================================
# BETTER TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir='./sahajbert_gold_model',

    eval_strategy='epoch',

    save_strategy='epoch',

    logging_strategy='epoch',

    learning_rate=1e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=10,

    weight_decay=0.01,

    warmup_ratio=0.10,

    load_best_model_at_end=True,

    metric_for_best_model='f1',

    greater_is_better=True,

    fp16=True,

    seed=SEED,

    report_to='none'
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

# ============================================================
# TRAIN
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\nTraining SahajBERT...")

trainer.train()

# ============================================================
# EVALUATE
# ============================================================

print("\nEvaluating on GOLD TEST SET...")

gold_results = trainer.evaluate(test_ds)

print("\nGold-only Results:")

for k, v in gold_results.items():
    print(f"{k}: {v}")

# ============================================================
# SAVE BEST MODEL
# ============================================================

trainer.save_model('./best_sahajbert_gold')

tokenizer.save_pretrained(
    './best_sahajbert_gold'
)

print("\n✅ Best model saved")

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]


Training SahajBERT...
{'loss': '0.7049', 'grad_norm': '25.56', 'learning_rate': '9.091e-06', 'epoch': '1'}
{'eval_loss': '0.6165', 'eval_accuracy': '0.7', 'eval_f1': '0.5263', 'eval_precision': '0.7812', 'eval_recall': '0.3968', 'eval_mcc': '0.3812', 'eval_runtime': '0.3183', 'eval_samples_per_second': '471.2', 'eval_steps_per_second': '31.41', 'epoch': '1'}


...

{'loss': '0.08674', 'grad_norm': '3.694', 'learning_rate': '5.657e-06', 'epoch': '5'}
{'eval_loss': '0.7064', 'eval_accuracy': '0.8133', 'eval_f1': '0.7455', 'eval_precision': '0.8723', 'eval_recall': '0.6508', 'eval_mcc': '0.6191', 'eval_runtime': '0.3171', 'eval_samples_per_second': '473.1', 'eval_steps_per_second': '31.54', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '24.42', 'train_samples_per_second': '286.7', 'train_steps_per_second': '18.02', 'train_loss': '0.3881', 'epoch': '5'}

Evaluating on GOLD TEST SET...
{'eval_loss': '0.5132', 'eval_accuracy': '0.7667', 'eval_f1': '0.7009', 'eval_precision': '0.7455', 'eval_recall': '0.6613', 'eval_mcc': '0.5132', 'eval_runtime': '0.3364', 'eval_samples_per_second': '446', 'eval_steps_per_second': '29.73', 'epoch': '5'}

Gold-only Results:
eval_loss: 0.5132082104682922
eval_accuracy: 0.7666666666666667
eval_f1: 0.7008547008547008
eval_precision: 0.7454545454545455
eval_recall: 0.6612903225806451
eval_mcc: 0.5131806052270687
eval_runtime: 0.3364
eval_samples_per_second: 445.953
eval_steps_per_second: 29.73
epoch: 5.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Best model saved


In [25]:
# ============================================================
# CELL 3 — LOAD 191K + APPLY EPH
# ============================================================

df_full = pd.read_csv(
    './BanglishDetect/data/raw/youtube_collected_raw.csv'
)

print(f"Raw comments: {len(df_full):,}")

print("\nApplying EPH preprocessing...")

texts = (
    df_full['text']
    .fillna('')
    .astype(str)
    .str[:1000]
    .tolist()
)

processed = []

for i, txt in enumerate(texts):

    if i % 5000 == 0:
        print(i)

    try:
        out = preprocess_P4_EPH(txt)

    except:
        out = ""

    processed.append(out)

df_full['text_eph'] = processed

df_full = df_full[
    df_full['text_eph'].str.strip().str.len() > 0
].reset_index(drop=True)

print(f"\nAfter EPH: {len(df_full):,}")

Raw comments: 191,000

Applying EPH preprocessing...
0
5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
55000
60000
65000
70000
75000
80000
85000
90000
95000
100000
105000
110000
115000
120000
125000
130000
135000
140000
145000
150000
155000
160000
165000
170000
175000
180000
185000
190000

After EPH: 190,992


In [26]:
# ============================================================
# CELL 4 — PSEUDO-LABEL 191K
# ============================================================

from tqdm import tqdm
import torch.nn.functional as F

DEVICE = torch.device(
    'cuda:0' if torch.cuda.is_available() else 'cpu'
)

texts_191k = (
    df_full['text_eph']
    .astype(str)
    .tolist()
)

print(f"\nPredicting on {len(texts_191k):,} comments")

# ============================================================
# LOAD TRAINED MODEL
# ============================================================

tok = AutoTokenizer.from_pretrained(
    './best_sahajbert_gold'
)

model = AutoModelForSequenceClassification.from_pretrained(
    './best_sahajbert_gold'
).to(DEVICE)

model.eval()

all_probs  = []
all_labels = []

BATCH = 64

for i in tqdm(range(0, len(texts_191k), BATCH)):

    batch_texts = texts_191k[i:i+BATCH]

    enc = tok(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors='pt'
    )

    enc = {
        k:v.to(DEVICE)
        for k,v in enc.items()
    }

    with torch.no_grad():

        outputs = model(**enc)

        probs = F.softmax(
            outputs.logits,
            dim=1
        )

        conf, pred = probs.max(dim=1)

    all_probs.extend(
        conf.cpu().numpy()
    )

    all_labels.extend(
        pred.cpu().numpy()
    )

df_full['pseudo_label'] = all_labels

df_full['pseudo_confidence'] = all_probs

# ============================================================
# KEEP ONLY TOP-K BEST PSEUDO LABELS
# ============================================================

TOP_K = 10000

df_pseudo = df_full.sort_values(
    'pseudo_confidence',
    ascending=False
).head(TOP_K).copy()

print(f"\nTop-K pseudo labels: {len(df_pseudo):,}")

print("\nPseudo-label distribution:")

print(df_pseudo['pseudo_label'].value_counts())

# SAVE
df_pseudo[
    [
        'text_eph',
        'pseudo_label',
        'pseudo_confidence'
    ]
].to_csv(
    './BanglishDetect/data/raw/pseudo_topk.csv',
    index=False
)

print("\n✅ Top-K pseudo-labeled dataset saved")


Predicting on 190,992 comments


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████| 2985/2985 [10:24<00:00,  4.78it/s]


Top-K pseudo labels: 10,000

Pseudo-label distribution:
pseudo_label
0    10000
Name: count, dtype: int64

✅ Top-K pseudo-labeled dataset saved


In [27]:
# ============================================================
# CELL 5 — CREATE IMPROVED COMBINED TRAIN SET
# ============================================================
# IMPORTANT:
# - only training receives pseudo labels
# - validation/test remain PURE HUMAN
# - human labels are UPSAMPLED
# ============================================================

df_pseudo = pd.read_csv(
    './BanglishDetect/data/raw/pseudo_topk.csv'
)

df_pseudo = df_pseudo.rename(columns={
    'text_eph': 'text',
    'pseudo_label': 'label'
})

# ============================================================
# GOLD DATA
# ============================================================

gold_train_df = pd.DataFrame({
    'text': X_train,
    'label': y_train,
    'source': 'human'
})

gold_val_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

gold_test_df = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

# ============================================================
# PSEUDO DATA
# ============================================================

df_pseudo['source'] = 'pseudo'

# ============================================================
# UPSAMPLE HUMAN LABELS
# ============================================================

HUMAN_BOOST = 5

boosted_human = pd.concat(
    [gold_train_df] * HUMAN_BOOST,
    ignore_index=True
)

combined_train_df = pd.concat([

    boosted_human,

    df_pseudo[
        ['text', 'label', 'source']
    ]

], ignore_index=True)

# ============================================================
# SHUFFLE
# ============================================================

combined_train_df = combined_train_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print(f"\nBoosted human samples: {len(boosted_human):,}")
print(f"Pseudo samples:        {len(df_pseudo):,}")
print(f"Combined train size:   {len(combined_train_df):,}")

print("\nSource distribution:")

print(combined_train_df['source'].value_counts())

print("\nLabel distribution:")

print(combined_train_df['label'].value_counts())

# SAVE
combined_train_df.to_csv(
    './BanglishDetect/data/processed/combined_train.csv',
    index=False
)

gold_val_df.to_csv(
    './BanglishDetect/data/processed/combined_val.csv',
    index=False
)

gold_test_df.to_csv(
    './BanglishDetect/data/processed/combined_test.csv',
    index=False
)

print("\n✅ Improved combined dataset saved")


Boosted human samples: 3,500
Pseudo samples:        10,000
Combined train size:   13,500

Source distribution:
source
pseudo    10000
human      3500
Name: count, dtype: int64

Label distribution:
label
0    12040
1     1460
Name: count, dtype: int64

✅ Improved combined dataset saved


In [30]:
# ============================================================
# CELL 6 — RETRAIN WITH IMPROVED SEMI-SUPERVISED SETUP
# ============================================================

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gc

# ============================================================
# LOAD COMBINED TRAIN DATA
# ============================================================

combined_train_df = pd.read_csv(
    './BanglishDetect/data/processed/combined_train.csv'
)

print(f"Combined training samples: {len(combined_train_df)}")

# ============================================================
# CREATE HF DATASET
# ============================================================

combined_train_ds = Dataset.from_pandas(
    combined_train_df[['text', 'label']]
)

# ============================================================
# TOKENIZE
# ============================================================

combined_train_ds = combined_train_ds.map(
    tokenize,
    batched=True
)

# ============================================================
# FORMAT FOR PYTORCH
# ============================================================

combined_train_ds.set_format(
    type='torch',
    columns=cols
)

# ============================================================
# FRESH MODEL
# ============================================================

model_combined = AutoModelForSequenceClassification.from_pretrained(
    SAHAJBERT,
    num_labels=2
)

model_combined.to(DEVICE)

# ============================================================
# COMPUTE CLASS WEIGHTS AUTOMATICALLY
# ============================================================

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=combined_train_df['label']
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
).to(DEVICE)

print(f"\nClass weights: {class_weights}")

# ============================================================
# WEIGHTED LOSS FUNCTION
# ============================================================

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)

# ============================================================
# CUSTOM TRAINER
# ============================================================

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):

        # Extract labels
        labels = inputs.pop("labels")

        # Forward pass
        outputs = model(**inputs)

        # Get logits
        logits = outputs.logits

        # Compute weighted CE loss
        loss = loss_fn(
            logits,
            labels
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

# ============================================================
training_args_combined = TrainingArguments(

    output_dir='./sahajbert_combined_model',

    # Evaluation / Saving
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',

    # Learning
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Epochs
    num_train_epochs=10,

    # Best model loading
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    # Mixed precision
    fp16=True,

    # Reproducibility
    seed=SEED,

    # Disable logging integrations
    report_to='none'
)

# ============================================================
# TRAINER
# ============================================================

trainer_combined = WeightedTrainer(

    model=model_combined,

    args=training_args_combined,

    train_dataset=combined_train_ds,

    eval_dataset=val_ds,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

# ============================================================
# CLEAN MEMORY
# ============================================================

gc.collect()
torch.cuda.empty_cache()

# ============================================================
# TRAIN
# ============================================================

print("\nTraining improved semi-supervised model...")

trainer_combined.train()

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\nEvaluating on ORIGINAL GOLD TEST SET...")

combined_results = trainer_combined.evaluate(
    test_ds
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\nImproved Combined Results:")

for k, v in combined_results.items():
    print(f"{k}: {v}")

print("\n✅ Improved semi-supervised experiment complete")

# ============================================================
# SAVE MODEL + TOKENIZER
# ============================================================

trainer_combined.save_model(
    './sahajbert_combined_model/final_model'
)

tokenizer.save_pretrained(
    './sahajbert_combined_model/final_model'
)

print("\n✅ Model saved successfully")

Combined training samples: 13500


Map:   0%|          | 0/13500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]


Class weights: tensor([0.5606, 4.6233], device='cuda:0')

Training improved semi-supervised model...
{'loss': '0.2932', 'grad_norm': '0.05911', 'learning_rate': '9.929e-06', 'epoch': '1'}
{'eval_loss': '1.11', 'eval_accuracy': '0.8467', 'eval_f1': '0.813', 'eval_precision': '0.8333', 'eval_recall': '0.7937', 'eval_mcc': '0.6838', 'eval_runtime': '0.312', 'eval_samples_per_second': '480.7', 'eval_steps_per_second': '32.05', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'loss': '6.025e-05', 'grad_norm': '0.0005138', 'learning_rate': '7.786e-06', 'epoch': '3'}
{'eval_loss': '2.179', 'eval_accuracy': '0.82', 'eval_f1': '0.7769', 'eval_precision': '0.8103', 'eval_recall': '0.746', 'eval_mcc': '0.628', 'eval_runtime': '0.3115', 'eval_samples_per_second': '481.6', 'eval_steps_per_second': '32.11', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '255.9', 'train_samples_per_second': '527.5', 'train_steps_per_second': '32.98', 'train_loss': '0.1073', 'epoch': '3'}

Evaluating on ORIGINAL GOLD TEST SET...
{'eval_loss': '1.512', 'eval_accuracy': '0.7667', 'eval_f1': '0.7154', 'eval_precision': '0.7213', 'eval_recall': '0.7097', 'eval_mcc': '0.5178', 'eval_runtime': '0.3155', 'eval_samples_per_second': '475.4', 'eval_steps_per_second': '31.7', 'epoch': '3'}

Improved Combined Results:
eval_loss: 1.5115684270858765
eval_accuracy: 0.7666666666666667
eval_f1: 0.7154471544715447
eval_precision: 0.7213114754098361
eval_recall: 0.7096774193548387
eval_mcc: 0.5177783498929814
eval_runtime: 0.3155
eval_samples_per_second: 475.433
eval_steps_per_second: 31.696
epoch: 3.0

✅ Improved semi-supervised experiment complete


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved successfully
